# TV-03 -- Internalisation du raisonnement (CoT -> calcul interne) -- v3 CoT supervisee + comparaison answer-only

Quatrieme tranche d'execution de l'Epic #17540 (Russell & Norvig arc B, raisonnement internalise). Cette v3 livre :

1. Le module tv/task.py etendu : nouvelles fonctions lot_multi_hop_cot(), evaluer_multi_hop_cot(), entrainer_cot(), entrainer_multi_seed_cot() qui implementent la supervision CoT (chain-of-thought) sur la tache multi-sauts.
2. Comparaison multi-seed (4 graines) answer-only vs CoT supervisee sur multi-sauts 3 sauts : mesure discriminante de l'internalisation du raisonnement (cf Huang et al. 2026).
3. Cycle de mesure complet : entrainement CoT avec perte supervisee sur la chaine entiere (chaque token PAS/RECAP_j/cible doit etre predit correctement).

## Pourquoi cette v3

v1 (c.808) posait le discriminant H.2 sur 1 graine. v2 (c.811) etendait a 4 graines et validait le protocole PR review-discipline §C (edge >= 2sigma cross-seed, 17.2sigma). Cette v3 livre la COMPARAISON discriminante : reponse only vs CoT supervisee, les deux sur la meme architecture MHA 64-dim.

## Ce qui n'est PAS dans cette tranche

- Lecture SAE des representations internes (autre lane po-2027:CoursIA).
- Modeles plus grands (la discrimination MHA 64-dim est suffisante pour la preuve de concept).
- Optimisation du vocabulaire CoT (les jetons PAS/RECAP_j sont des placeholders pedagogiques ; un tokenization appris est un autre grain).


In [1]:
import math
import sys
import time

import torch
import torch.nn.functional as F

sys.path.insert(0, '.')
from tv import (  # noqa: E402
    PetitLM, Vocab,
    evaluer_single_hop, evaluer_multi_hop, evaluer_multi_hop_cot,
    entrainer, entrainer_multi_seed,
    entrainer_cot, entrainer_multi_seed_cot,
)

print(f'torch {torch.__version__} | python {sys.version_info.major}.{sys.version_info.minor}')
print('package tv charge OK : single-hop + multi-hop + multi-hop-CoT + multi-seed')


torch 2.13.0+cu126 | python 3.13
package tv charge OK : single-hop + multi-hop + multi-hop-CoT + multi-seed


## 1. Vocabulaires

Deux vocabulaires de même structure :

- **Single-hop** : `N_MARQUEURS=8`, `N_REMPLISSAGE=10`, `N_QUESTIONS=1` (le `QUESTION(0)` trivial). Vocab total 20 tokens.
- **Multi-sauts** : `N_MARQUEURS=8`, `N_REMPLISSAGE=10`, `N_QUESTIONS=3` (la cible est le `q`-ième marqueur, avec `q` choisi uniformément sur [0, 3)). Vocab total 22 tokens.

Hasard exactitude : 1/8 dans les deux cas (la cible est l'un des 8 marqueurs ; le mécanisme à apprendre est la sélection conditionnelle, pas la mémorisation).

In [2]:
T = 64  # longueur de sequence commune aux deux taches
T_COT = T + 7  # 7 = 2*max_q+1 = 2*3+1 (chaine CoT)
v_single = Vocab(N_MARQUEURS=8, N_REMPLISSAGE=10, N_QUESTIONS=1)
v_multi = Vocab(N_MARQUEURS=8, N_REMPLISSAGE=10, N_QUESTIONS=3)

MAX_Q = 3  # q_idx in [0, 3) -- borne sup pour le vocabulaire CoT
VOCAB_COT = v_multi.VOCAB + 1 + MAX_Q  # VOCAB + JETON_PAS + MAX_Q jetons de recap

def fabrique(vocab):
    """Fabrique un MHA vierge pour taches sans CoT (VOCAB standard)."""
    return PetitLM(
        vocab=vocab.VOCAB,
        d_model=64,
        n_heads=4,
        n_kv_heads=4,
        window=None,
        n_couches=2,
    )

def fabrique_cot(vocab):
    """Fabrique un MHA vierge pour tache CoT (VOCAB etendu avec PAS + RECAP)."""
    return PetitLM(
        vocab=VOCAB_COT,
        d_model=64,
        n_heads=4,
        n_kv_heads=4,
        window=None,
        n_couches=2,
    )

print(f'Vocab single-hop : VOCAB={v_single.VOCAB}, N_QUESTIONS={v_single.N_QUESTIONS}')
print(f'Vocab multi-sauts : VOCAB={v_multi.VOCAB}, N_QUESTIONS={v_multi.N_QUESTIONS}')
print(f'VOCAB_COT (multi-sauts + CoT) : {VOCAB_COT} (= {v_multi.VOCAB} + 1 JETON_PAS + {MAX_Q} jetons RECAP)')
print(f'Hasard exactitude : 1 / {v_single.N_MARQUEURS} = {1/v_single.N_MARQUEURS:.4f}')


Vocab single-hop : VOCAB=20, N_QUESTIONS=1
Vocab multi-sauts : VOCAB=22, N_QUESTIONS=3
VOCAB_COT (multi-sauts + CoT) : 26 (= 22 + 1 JETON_PAS + 3 jetons RECAP)
Hasard exactitude : 1 / 8 = 0.1250


## 2. Mesure multi-seed single-hop (4 graines, 300 pas)

Single-hop est trivialement resolue par MHA 102K params : on s'attend a 1.0000 +/- ~0 sur 4 graines.


In [3]:
GRAINES = [0, 1, 7, 42]
PAS = 300

result_s = entrainer_multi_seed(
    fabrique, v_single, T=T, multi_hop=False, graines=GRAINES, pas=PAS
)
print(f'SINGLE-HOP multi-seed ({len(GRAINES)} graines, {PAS} pas) :')
print(f'  EXACTITUDE = {result_s["acc_moy"]:.4f} +/- {result_s["acc_std"]:.4f}  (hasard = 0.1250)')
print(f'  PERPLEXITE = {result_s["ppl_moy"]:.4f} +/- {result_s["ppl_std"]:.4f}  (hasard = 8.0)')
print(f'  Secondes total = {result_s["secondes"]:.2f} s')
for graine, acc, ppl, sec in result_s["brut"]:
    print(f'    graine {graine} : acc={acc:.4f} ppl={ppl:.4f} sec={sec:.2f}')


SINGLE-HOP multi-seed (4 graines, 300 pas) :
  EXACTITUDE = 1.0000 +/- 0.0000  (hasard = 0.1250)
  PERPLEXITE = 1.0014 +/- 0.0001  (hasard = 8.0)
  Secondes total = 48.56 s
    graine 0 : acc=1.0000 ppl=1.0013 sec=11.66
    graine 1 : acc=1.0000 ppl=1.0015 sec=11.85
    graine 7 : acc=1.0000 ppl=1.0015 sec=10.54
    graine 42 : acc=1.0000 ppl=1.0015 sec=11.53


## 3. Mesure multi-seed multi-sauts (4 graines, 300 pas)

Multi-sauts 3 questions : le discriminant H.2 doit tenir sur 4 graines (moyenne significativement au-dessus du hasard, sans atteindre 1.0).


In [4]:
result_m = entrainer_multi_seed(
    fabrique, v_multi, T=T, multi_hop=True, graines=GRAINES, pas=PAS
)
print(f'MULTI-SAUTS multi-seed ({len(GRAINES)} graines, {PAS} pas, 3 sauts) :')
print(f'  EXACTITUDE = {result_m["acc_moy"]:.4f} +/- {result_m["acc_std"]:.4f}  (hasard = 0.1250)')
print(f'  PERPLEXITE = {result_m["ppl_moy"]:.4f} +/- {result_m["ppl_std"]:.4f}  (hasard = 8.0)')
print(f'  Secondes total = {result_m["secondes"]:.2f} s')
for graine, acc, ppl, sec in result_m["brut"]:
    print(f'    graine {graine} : acc={acc:.4f} ppl={ppl:.4f} sec={sec:.2f}')


MULTI-SAUTS multi-seed (4 graines, 300 pas, 3 sauts) :
  EXACTITUDE = 0.4019 +/- 0.0161  (hasard = 0.1250)
  PERPLEXITE = 2.9156 +/- 0.0943  (hasard = 8.0)
  Secondes total = 42.33 s
    graine 0 : acc=0.3945 ppl=2.9109 sec=10.38
    graine 1 : acc=0.4180 ppl=3.0707 sec=9.89
    graine 7 : acc=0.4160 ppl=2.8310 sec=10.14
    graine 42 : acc=0.3789 ppl=2.8499 sec=11.46


## 4. Bilan tranche v2

Multi-seed >=4 sur single-hop vs multi-sauts -- discriminant H.2 mesure sur 4 graines.


In [5]:
print('BILAN multi-seed :')
print(f'  single-hop : {result_s["acc_moy"]:.4f} +/- {result_s["acc_std"]:.4f}')
print(f'  multi-sauts : {result_m["acc_moy"]:.4f} +/- {result_m["acc_std"]:.4f}')
rapport = result_m["acc_moy"] / result_s["acc_moy"] if result_s["acc_moy"] > 0 else float("inf")
print(f'  rapport multi/single = {rapport:.3f}')
print(f'  secondes total (4 graines single + 4 graines multi) = {result_s["secondes"] + result_m["secondes"]:.2f} s')
print()
if result_s['acc_moy'] >= 0.99 and result_m['acc_moy'] < result_s['acc_moy']:
    print('Conclusion : Single-hop trivialement resolu (>=0.99) ; multi-sauts plus bas avec >=4 graines.')
    print('Discriminant H.2 multi-seed : OK (single >> multi, multi > hasard).')
else:
    print(f'ATTENTION : single-hop = {result_s["acc_moy"]:.4f}, multi-sauts = {result_m["acc_moy"]:.4f}.')
    print('Le discriminant H.2 n est PAS clairement tenu. Investiguer la graine fautive.')


BILAN multi-seed :
  single-hop : 1.0000 +/- 0.0000
  multi-sauts : 0.4019 +/- 0.0161
  rapport multi/single = 0.402
  secondes total (4 graines single + 4 graines multi) = 90.89 s

Conclusion : Single-hop trivialement resolu (>=0.99) ; multi-sauts plus bas avec >=4 graines.
Discriminant H.2 multi-seed : OK (single >> multi, multi > hasard).


## 5. Comparaison CoT supervisee vs answer-only (4 graines, 300 pas)

Le discriminant central du programme #17540 : la supervision CoT ameliore-t-elle significativement la memorisation multi-sauts ?

- answer-only (v2 cellule 7) : `entrainer_multi_seed` avec `multi_hop=True` (cible = marqueur direct)
- CoT supervise (v3) : `entrainer_multi_seed_cot` avec `max_q=3` (cible finale + chaine PAS/RECAP)

Tell c.1493 strict fondateur nuance : la mesure CoT supervise la chaine entiere via une perte massee par item (les jetons au-dela du q_idx reel sont ignores ; voir entrainer_cot dans tv/task.py).

In [6]:
result_cot = entrainer_multi_seed_cot(
    fabrique_cot, v_multi, T_cot=T_COT, graines=GRAINES, pas=PAS, batch=32, max_q=MAX_Q
)
print(f'CoT multi-seed ({len(GRAINES)} graines, {PAS} pas, 3 sauts) :')
print(f'  EXACTITUDE = {result_cot["acc_moy"]:.4f} +/- {result_cot["acc_std"]:.4f}  (hasard = 0.1250)')
print(f'  PERPLEXITE = {result_cot["ppl_moy"]:.4f} +/- {result_cot["ppl_std"]:.4f}  (hasard = 8.0)')
print(f'  Secondes total = {result_cot["secondes"]:.2f} s')
for graine, acc, ppl, sec in result_cot["brut"]:
    print(f'    graine {graine} : acc={acc:.4f} ppl={ppl:.4f} sec={sec:.2f}')


CoT multi-seed (4 graines, 300 pas, 3 sauts) :
  EXACTITUDE = 0.0000 +/- 0.0000  (hasard = 0.1250)
  PERPLEXITE = 17994.0545 +/- 12288.0419  (hasard = 8.0)
  Secondes total = 47.90 s
    graine 0 : acc=0.0000 ppl=36505.7196 sec=13.78
    graine 1 : acc=0.0000 ppl=3746.3259 sec=11.59
    graine 7 : acc=0.0000 ppl=10901.5518 sec=10.92
    graine 42 : acc=0.0000 ppl=20822.6206 sec=11.13


## 6. Bilan tranche v3 -- comparaison discriminante

Compare answer-only (multi-hop) vs CoT supervisee (multi-hop + chain-of-thought) sur la meme architecture MHA 64-dim et 4 graines communes.

In [7]:
print('BILAN v3 comparaison answer-only vs CoT supervisee :')
print(f'  answer-only  : {result_m["acc_moy"]:.4f} +/- {result_m["acc_std"]:.4f}  (sec {result_m["secondes"]:.1f})')
print(f'  CoT superv.  : {result_cot["acc_moy"]:.4f} +/- {result_cot["acc_std"]:.4f}  (sec {result_cot["secondes"]:.1f})')
print()
rapport_cot = result_cot['acc_moy'] / max(result_m['acc_moy'], 1e-9)
print(f'  Rapport CoT / answer-only = {rapport_cot:.3f}')
print()
if result_cot['acc_moy'] > result_m['acc_moy'] + 2 * max(result_m['acc_std'], 0.01):
    print('Conclusion : CoT supervisee > answer-only avec marge > 2*std.')
    print('Discriminant central valide : la supervision de la chaine aide le modele a internaliser le comptage/selection multi-sauts.')
elif result_cot['acc_moy'] > result_m['acc_moy']:
    print(f'Conclusion : CoT supervisee > answer-only en moyenne (mais marge < 2*std).')
    print('Tendance favorable ; grain suivant = augmenter pas ou evaluer sur davantage de graines.')
elif result_cot['acc_moy'] < result_m['acc_moy'] - 0.05:
    print(f'Conclusion : CoT supervisee < answer-only de plus de 5 points.')
    print('Surprenant -- le modele struggle a generer la chaine entiere en 300 pas. Investiguer.')
else:
    print(f'Conclusion : CoT supervisee ~ answer-only.')
    print('Les deux regimes convergent au meme endroit ; la discrimination necessite probablement plus de pas ou un modele plus gros.')


BILAN v3 comparaison answer-only vs CoT supervisee :
  answer-only  : 0.4019 +/- 0.0161  (sec 42.3)
  CoT superv.  : 0.0000 +/- 0.0000  (sec 47.9)

  Rapport CoT / answer-only = 0.000

Conclusion : CoT supervisee < answer-only de plus de 5 points.
Surprenant -- le modele struggle a generer la chaine entiere en 300 pas. Investiguer.
